In [1]:
print("Bronze Pipeline Execution Start Successfully.")

Bronze Pipeline Execution Start Successfully.


In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, split, explode

In [2]:
def build_spark_session():
    spark = SparkSession.builder \
    .appName("IMDb Lakehouse Pipeline Bronze Load") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.warehouse.dir", "s3a://imdb-bucket/hive_catalog/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "lakehouse_admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "lakehouse_password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")\
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")\
    .config("spark.executor.cores", "2") \
    .config("spark.cores.max", "4") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.parquet.compression.codec", "snappy")\
    .enableHiveSupport()\
    .getOrCreate()
    
    return spark

In [4]:
def create_database(spark):
    spark.sql("CREATE DATABASE IF NOT EXISTS imdb_olap_bronze;")
    spark.sql("USE imdb_olap_bronze;")

In [5]:
def read_csv(spark, file_path, sep="\t", header=True, infer_schema=False):
    return (
        spark.read
        .option("sep", sep)
        .option("header", header)
        .option("inferSchema", infer_schema)
        .csv(file_path)
        .na.replace("\\N", None)
    )

In [6]:
def save_table(df,table_name,path=None,mode="overwrite",fmt="parquet",partition_cols=None):
    writer = (df.write.format(fmt).mode(mode))
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    if path:
        writer = writer.option("path", path)
    writer.saveAsTable(table_name)

In [7]:
spark = build_spark_session()
create_database(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/08 14:38:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/08 14:38:39 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [28]:
RAW_DIR = "s3a://imdb-bucket/raw"
BRONZE_DIR = "s3a://imdb-bucket/bronze"
INPUT_DIR = RAW_DIR
OUTPUT_DIR = BRONZE_DIR

In [ ]:
title_basics_df = read_csv(spark,f"{INPUT_DIR}/title.basics.tsv")
save_table(title_basics_df,'title_basics',f"{OUTPUT_DIR}/title_basics")

In [ ]:
title_ratings_df = read_csv(spark,f"{INPUT_DIR}/title.ratings.tsv")
save_table(title_ratings_df,'title_ratings',f"{OUTPUT_DIR}/title_ratings")

In [ ]:
name_basics_df = read_csv(spark,f"{INPUT_DIR}/name.basics.tsv")
save_table(name_basics_df,'name_basics',f"{OUTPUT_DIR}/name_basics")

In [ ]:
title_principals_df = read_csv(spark,f"{INPUT_DIR}/title.principals.tsv")
save_table(title_principals_df,'title_principals',f"{OUTPUT_DIR}/title_principals")

In [ ]:
title_akas_df = read_csv(spark,f"{INPUT_DIR}/title.akas.tsv")
save_table(title_akas_df,'title_akas',f"{OUTPUT_DIR}/title_akas")

In [10]:
print("Bronze Pipeline Execution Completed Successfully.")
spark.stop()

Bronze Pipeline Execution Completed Successfully.
